# Expected Move

## Expected Move Calculation

Expected Move = Stock Price × Implied Volatility × √(Days to Expiry / 365)

In [ ]:
import math
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import mplfinance as mpf
import numpy as np
import pandas as pd
import yfinance as yf

In [ ]:
def calculate_expected_move(current_price, vix_value, days_to_expiry=1):
    """
    Calculate expected move using VIX implied volatility

    Parameters:
    current_price: Current SPX price
    vix_value: Current VIX value
    days_to_expiry: Number of days for the expected move (default 1 day)

    Returns:
    expected_move: Expected move in points
    """
    # Convert VIX to decimal (VIX is quoted as percentage)
    # implied_vol = vix_value / 100
    implied_vol = 0.2020

    # Calculate expected move using the formula:
    # Expected Move = Stock Price × Implied Volatility × √(Days to Expiry / 365)
    expected_move = current_price * implied_vol * math.sqrt(days_to_expiry / 365)

    return expected_move

In [ ]:
calculate_expected_move(5980.87, 20.58, 20)

In [ ]:
def get_market_data(period="1y"):
    """
    Fetch SPX and VIX data from Yahoo Finance
    """
    try:
        # Fetch SPX data (using SPY as proxy since ^GSPC sometimes has issues)
        spx = yf.download("^GSPC", period=period, interval="1d", auto_adjust=True)

        # Flatten multi-index columns
        if isinstance(spx.columns, pd.MultiIndex):
            column_names = ["Date"] + spx.columns.get_level_values(0).tolist()
            spx = spx.reset_index()
            spx.columns = column_names
            spx = spx.set_index("Date")

        # Fetch VIX data
        vix = yf.download("^VIX", period=period, interval="1d", auto_adjust=True)

        # Flatten multi-index columns
        if isinstance(vix.columns, pd.MultiIndex):
            column_names = ["Date"] + vix.columns.get_level_values(0).tolist()
            vix = vix.reset_index()
            vix.columns = column_names
            vix = vix.set_index("Date")

        return spx, vix
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None, None

In [ ]:
def get_expected_move_cone(spot, volatility, last_date, num_days=30):
    """
    Generate a DataFrame with expected move cone (1-sigma and 2-sigma bands) for forward-looking dates.
    Returns a DataFrame indexed by date with columns: 'upper_1', 'lower_1', 'upper_2', 'lower_2'.
    """
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1), periods=num_days, freq="D"
    )
    days_forward = np.arange(1, num_days + 1)
    expected_moves = [
        calculate_expected_move(spot, volatility, d) for d in days_forward
    ]
    upper_1 = spot + np.array(expected_moves)
    lower_1 = spot - np.array(expected_moves)
    upper_2 = spot + 2 * np.array(expected_moves)
    lower_2 = spot - 2 * np.array(expected_moves)
    df = pd.DataFrame(
        {
            "upper_1": upper_1,
            "lower_1": lower_1,
            "upper_2": upper_2,
            "lower_2": lower_2,
        },
        index=future_dates,
    )
    return df

In [ ]:
def print_expected_moves_table(current_spx, current_vix, cone_df):
    """
    Print a table of expected moves for each forward day using the cone dataframe.
    """
    print("\n" + "=" * 50)
    print("EXPECTED MOVE ANALYSIS")
    print("=" * 50)
    print(f"Current SPX Price: {current_spx:.2f}")
    print(f"Current VIX Level: {current_vix:.2f}")
    print("\nExpected Moves:")
    print("-" * 30)
    for i, (date, row) in enumerate(cone_df.iterrows(), 1):
        move = (row["upper_1"] - row["lower_1"]) / 2
        move_pct = (move / current_spx) * 100
        print(f"{i:2d} Day(s) [{date.date()}]: ±{move:6.1f} points (±{move_pct:.1f}%)")
        print(f"         1σ Range: {row['lower_1']:.1f} - {row['upper_1']:.1f}")
        print(f"         2σ Range: {row['lower_2']:.1f} - {row['upper_2']:.1f}")
        print()

In [ ]:
def print_expected_moves(spx_data, vix_data, days_ahead=30):
    if spx_data is None or vix_data is None:
        print("No data available for plotting")
        return

    # Get the latest values
    current_spx = spx_data["Close"].iloc[-1]
    current_vix = vix_data["Close"].iloc[-1]
    last_date = spx_data.index[-1]

    # Get expected move cone DataFrame
    cone_df = get_expected_move_cone(
        current_spx, current_vix, last_date, num_days=days_ahead
    )

    # Print expected moves table
    print_expected_moves_table(current_spx, current_vix, cone_df)

In [ ]:
def plot_expected_move_chart(spx_data, vix_data, days_ahead=30):
    """
    Create a chart showing SPX price with expected move bands using the cone dataframe
    """
    if spx_data is None or vix_data is None:
        print("No data available for plotting")
        return

    # Get the latest values
    current_spx = spx_data["Close"].iloc[-1]
    current_vix = vix_data["Close"].iloc[-1]
    last_date = spx_data.index[-1]

    # Get expected move cone DataFrame
    cone_df = get_expected_move_cone(
        current_spx, current_vix, last_date, num_days=days_ahead
    )

    # Prepare data for plotting
    all_dates = list(spx_data.index) + list(cone_df.index)
    all_prices = list(spx_data["Close"]) + [np.nan] * len(cone_df)
    upper_full_1 = [np.nan] * len(spx_data) + list(cone_df["upper_1"])
    lower_full_1 = [np.nan] * len(spx_data) + list(cone_df["lower_1"])
    upper_full_2 = [np.nan] * len(spx_data) + list(cone_df["upper_2"])
    lower_full_2 = [np.nan] * len(spx_data) + list(cone_df["lower_2"])

    # Create the main plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Plot SPX price
    ax1.plot(spx_data.index, spx_data["Close"], label="SPX", color="blue", linewidth=2)

    # Plot the 2-sigma expected move cone (lighter fill)
    ax1.plot(
        all_dates,
        upper_full_2,
        color="skyblue",
        linestyle="--",
        label="2 Std Dev Upper",
    )
    ax1.plot(
        all_dates,
        lower_full_2,
        color="skyblue",
        linestyle="--",
        label="2 Std Dev Lower",
    )
    ax1.fill_between(
        all_dates,
        upper_full_2,
        lower_full_2,
        color="skyblue",
        alpha=0.15,
        label="2 Std Dev Cone",
    )

    # Plot the 1-sigma expected move cone
    ax1.plot(
        all_dates, upper_full_1, color="blue", linestyle="--", label="1 Std Dev Upper"
    )
    ax1.plot(
        all_dates, lower_full_1, color="blue", linestyle="--", label="1 Std Dev Lower"
    )
    ax1.fill_between(
        all_dates,
        upper_full_1,
        lower_full_1,
        color="blue",
        alpha=0.2,
        label="1 Std Dev Cone",
    )

    ax1.set_title(
        f"SPX Expected Move Analysis\nCurrent Price: {current_spx:.2f}, VIX: {current_vix:.2f}",
        fontsize=14,
        fontweight="bold",
    )
    ax1.set_ylabel("SPX Price")
    ax1.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax1.grid(True, alpha=0.3)

    # Plot VIX
    ax2.plot(vix_data.index, vix_data["Close"], label="VIX", color="red", linewidth=2)
    ax2.set_title("VIX (Implied Volatility)", fontsize=12, fontweight="bold")
    ax2.set_ylabel("VIX Level")
    ax2.set_xlabel("Date")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def create_mplfinance_chart(spx_data, vix_data):
    """
    Create a candlestick chart using mplfinance with expected move cone for 30 forward-looking days, using native mplfinance addplot and fill_between.
    """
    if spx_data is None or vix_data is None:
        print("No data available for mplfinance chart")
        return

    # Get current values
    current_spx = spx_data["Close"].iloc[-1]
    current_vix = vix_data["Close"].iloc[-1]
    last_date = spx_data.index[-1]

    # Get expected move cone DataFrame
    cone_df = get_expected_move_cone(current_spx, current_vix, last_date, num_days=30)

    # Prepare combined DataFrame for plotting
    # spx_tail = spx_data.tail(20).copy()
    spx_tail = spx_data.copy()
    # Create a DataFrame with the same columns as spx_tail, indexed by cone_df's index, filled with NaN
    nan_df = pd.DataFrame(np.nan, index=cone_df.index, columns=spx_tail.columns)
    combined = pd.concat([spx_tail, nan_df])

    # Add the cone bands to the combined DataFrame
    combined["upper_1"] = pd.concat(
        [pd.Series([np.nan] * len(spx_tail), index=spx_tail.index), cone_df["upper_1"]]
    )
    combined["lower_1"] = pd.concat(
        [pd.Series([np.nan] * len(spx_tail), index=spx_tail.index), cone_df["lower_1"]]
    )
    combined["upper_2"] = pd.concat(
        [pd.Series([np.nan] * len(spx_tail), index=spx_tail.index), cone_df["upper_2"]]
    )
    combined["lower_2"] = pd.concat(
        [pd.Series([np.nan] * len(spx_tail), index=spx_tail.index), cone_df["lower_2"]]
    )

    # Prepare addplots for 1-sigma and 2-sigma bands
    addplot = [
        mpf.make_addplot(
            combined["upper_2"],
            color="skyblue",
            linestyle="--",
            width=1,
            label="2 Std Dev Upper",
        ),
        mpf.make_addplot(
            combined["lower_2"],
            color="skyblue",
            linestyle="--",
            width=1,
            label="2 Std Dev Lower",
        ),
        mpf.make_addplot(
            combined["upper_1"],
            color="blue",
            linestyle="--",
            width=1,
            label="1 Std Dev Upper",
        ),
        mpf.make_addplot(
            combined["lower_1"],
            color="blue",
            linestyle="--",
            width=1,
            label="1 Std Dev Lower",
        ),
    ]

    # Use fill_between to fill the cones
    fill_between = [
        dict(
            y1=combined["upper_2"].values,
            y2=combined["lower_2"].values,
            color="skyblue",
            alpha=0.15,
            label="2 Std Dev Cone",
        ),
        dict(
            y1=combined["upper_1"].values,
            y2=combined["lower_1"].values,
            color="blue",
            alpha=0.2,
            label="1 Std Dev Cone",
        ),
    ]

    # Plot with mplfinance
    mpf.plot(
        combined,
        type="ohlc",
        style="charles",
        title=f"SPX with 30-Day Expected Move Cone\nCurrent Price: {current_spx:.2f}, VIX: {current_vix:.2f}",
        volume=False,
        addplot=addplot,
        fill_between=fill_between,
        figsize=(14, 8),
        show_nontrading=True,
        ylabel="SPX Price",
        tight_layout=True,
        returnfig=False,
    )

In [ ]:
def main():
    """
    Main function to run the expected move analysis
    """
    print("Fetching SPX and VIX data...")

    # Get market data
    spx_data, vix_data = get_market_data(period="1y")

    if spx_data is None or vix_data is None:
        print("Failed to fetch market data. Please check your internet connection.")
        return

    print("Data fetched successfully!")

    # Plot expected move analysis
    plot_expected_move_chart(spx_data, vix_data)

    # Create mplfinance candlestick chart
    create_mplfinance_chart(spx_data, vix_data)

    # Print expected moves table
    print_expected_moves(spx_data, vix_data)

In [ ]:
spx_data, vix_data = get_market_data(period="1y")

In [ ]:
spx_data.head()

In [ ]:
spx_data.tail()

In [ ]:
vix_data.tail()

In [ ]:
plot_expected_move_chart(spx_data, vix_data)

In [ ]:
# Create mplfinance candlestick chart
create_mplfinance_chart(spx_data, vix_data)

In [ ]:
print_expected_moves(spx_data, vix_data)